In [ ]:
from langchain_community.document_loaders import PyPDFLoader,TextLoader
from langchain_text_splitters import NLTKTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import nltk
from langchain_openai import ChatOpenAI
nltk.download("punkt")
from nltk.tokenize import sent_tokenize

c:\Users\USER\Documents\machine-learning\projects\slgs\rag-poc\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [73]:
from langchain_aws import ChatBedrock
from dotenv import load_dotenv
from langchain_community.llms import Ollama

load_dotenv(".env",override=True)

os.environ["BEDROCK_API_KEY"] = os.getenv("BEDROCK_API_KEY")
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"] = os.getenv("AWS_DEFAULT_REGION")


vlm = ChatBedrock(
    model_id="arn:aws:bedrock:ap-southeast-1:408897322877:application-inference-profile/pf3b7fwx6on0",
    region_name="ap-southeast-1",
    provider="anthropic",

)

model = Ollama(model="gemma:2b")
# openai_api_key = os.getenv("OPENAI_API_KEY")
# model = ChatOpenAI(model="gpt-4o-mini", api_key=openai_api_key,temperature=0)
model
vlm

ChatBedrock(client=<botocore.client.BedrockRuntime object at 0x0000021592362270>, bedrock_client=<botocore.client.Bedrock object at 0x00000215923630E0>, region_name='ap-southeast-1', aws_access_key_id=SecretStr('**********'), aws_secret_access_key=SecretStr('**********'), provider='anthropic', model_id='arn:aws:bedrock:ap-southeast-1:408897322877:application-inference-profile/pf3b7fwx6on0', base_model_id='anthropic.claude-opus-4-5-20251101-v1:0', model_kwargs={}, profile={})

In [39]:
from langchain_core.messages import HumanMessage

def _invoke_bedrock_image(image_b64, mime_type):
    message = HumanMessage(
        content=[
            {
                "type": "text",
                "text": (
                    "You are an expert AI assistant, you are tasked with extracting the entire text from any PDF document. The document can be simple, complex, or even scanned, this shouldn't matter to you."
                    "You will be given the entire PDF as input. Start examining the document page by page, when you come across text, extract it as is don't convert it into another format like HTML or Markdown. If you come across images, replace them with a very detailed description of the image while taking into consideration the context around it."
                    "When you come across tables, describe them too like the image. The description should be very detailed and in a way that someone will understand the table without seeing it."
                    "Make sure to keep the structure of the document, if there are sections, subsections, bullet points, or numbered lists, make sure to keep them as is. If there are any headers, footers, page numbers, remove them."
                    "If there are line breaks mid-sentence, please remove it and display the entire sentence or paragraph in smooth flow."
                    "The final output should be a clean, well-structured text that represents the content of the entire PDF document as closely as possible to how a human would see it with their eyes when reading the document. Don't say anything else, just output the text you extracted from the PDF."
                    "Here is the PDF:"
                ),
            },
            {
                "type": "image",
                "source": {
                    "type": "base64",
                    "media_type": mime_type,
                    "data": image_b64,
                },
            },
        ]
    )

    return vlm.invoke([message]) 

In [40]:
import base64
def _encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode()

In [41]:
import io
import base64
import mimetypes
from pdf2image import convert_from_path

def parse_document(image_path):

    print(f"IMAGE PATH: {image_path}")


    if not image_path:
        return None
    
    if not os.path.exists(image_path):
        raise ValueError(f"Image not found: {image_path}")
    
    mime_type, _ = mimetypes.guess_type(image_path)
    results = []

    if mime_type in ["image/png", "image/jpeg", "image/webp"]:
        image_b64 = _encode_image(image_path)
        response = _invoke_bedrock_image(image_b64, mime_type)
        results.append(response.content)

    elif mime_type == "application/pdf":
        pages = convert_from_path(
            image_path,
            dpi=200,
            fmt="png",
            poppler_path=r"C:\poppler-25.12.0\Library\bin"
        )

        for i, page in enumerate(pages):
            buffer = io.BytesIO()
            page.save(buffer, format="PNG")


            image_b64 = base64.b64encode(buffer.getvalue()).decode()

            response = _invoke_bedrock_image(image_b64,"image/png")

            print(f"RESPONSE: {response}")

            results.append(response.content)
    else:
        raise ValueError(f"Unsupported file type: {mime_type}")
    
    return results

In [42]:

def analyze_and_convert_document(file_path: str, filename: str):

    PATH = "http://localhost:5001/api/v1/documents/uploads"
    UPLOAD_FOLDER = "app/uploads"

    os.makedirs(UPLOAD_FOLDER, exist_ok=True)
    md_filename = f"{UPLOAD_FOLDER}/{filename}.md"
    response = parse_document(file_path)

    for r in response:
        try:
            # Open the file in append mode ('a')
            with open(md_filename, "a", encoding="utf-8") as file:
                # Append content to the file
                file.write(r)
            print(f"Appended to {md_filename} successfully.")

        except Exception as e:
            print(f"An error occurred: {e}")

    return md_filename

In [ ]:
loader = TextLoader(
    analyze_and_convert_document("D:/Users/RouVin/Downloads/DEVELOPMENT PLAN - TOUR EAST FANTASY.pdf", "development_plan_tour_east_fantasy.md"),
    encoding="utf-8"
)
import json

documents = loader.load()
# for d in documents:
#     print(d.page_content)
documents[0].page_content

IMAGE PATH: D:/Users/RouVin/Downloads/DEVELOPMENT PLAN - TOUR EAST FANTASY.pdf
RESPONSE: content='DEVELOPMENT OF TOUR EAST FANTASY\n\nTour East Fantasy is a Tourism and Hospitality super app, providing tourists with ease of tourism information, promotions, and ecommerce capability to easily purchase tourism related products and services. Think of it as a Grab App for Tourism.\n\nThere will be 2 components to this platform\n\n1. Tour East Fantasy 3js experience – This is the wow factor used for promoting the platform and to get partners on board. It\'s friendly and easy to navigate, to get visitors to explore and navigate. https://toureast.xctualyfe.com/\n\n2. Tour East Fantasy Web App – This will encapsulate the entire experience and will lead the user to finding other products and services, as well as play hyper casual games to win prizes.\n\nA table with three columns labeled "Phase", "Key Activity", and "Duration". The table contains three rows of data:\n\nPhase 1 contains five bull

'DEVELOPMENT OF TOUR EAST FANTASY\n\nTour East Fantasy is a Tourism and Hospitality super app, providing tourists with ease of tourism information, promotions, and ecommerce capability to easily purchase tourism related products and services. Think of it as a Grab App for Tourism.\n\nThere will be 2 components to this platform\n\n1. Tour East Fantasy 3js experience – This is the wow factor used for promoting the platform and to get partners on board. It\'s friendly and easy to navigate, to get visitors to explore and navigate. https://toureast.xctualyfe.com/\n\n2. Tour East Fantasy Web App – This will encapsulate the entire experience and will lead the user to finding other products and services, as well as play hyper casual games to win prizes.\n\nA table with three columns labeled "Phase", "Key Activity", and "Duration". The table contains three rows of data:\n\nPhase 1 contains the following key activities as bullet points: Project Kick Off; Develop a web version of Tour East Fantas

In [ ]:
import os
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
llm = ChatOpenAI(model="gpt-4o-mini", api_key=openai_api_key,temperature=0)

chunks = []

class ChunkMeta(BaseModel):
    title: str = Field(description="The title of the chunk.")
    summary: str = Field(description="The summary of the chunk.")
    
def create_new_chunk(chunk_id, proposition):
    summary_llm = llm.with_structured_output(ChunkMeta)

    summary_prompt_template = ChatPromptTemplate.from_messages([
        (
            "system",
            """Generate BOTH:
                - a title
                - a summary

                Always return both fields.

            """,
        ),
        (
            "user",
            "propositions:{propositions}"
        )
    ])

    summary_chain = summary_prompt_template | summary_llm

    chunk_meta = summary_chain.invoke(
        {
            "propositions": [proposition]
        }
    )
    
    chunk = {
        "id": chunk_id,
        "summary": chunk_meta.summary,
        "title": chunk_meta.title,
        "propositions": [proposition],
    }

    chunks.append(chunk)
    print(f"CHUNKS: {chunks}")

    return chunk_id




def add_proposition(chunk_id, proposition):

    # print(f"CHUNK ID: {chunk_id}")
    # print(f"PROPOSITION: {proposition}")
    summary_llm = llm.with_structured_output(ChunkMeta)

    summary_prompt_template = ChatPromptTemplate.from_messages([
        (
            "system",
            "If the current_summary and title is still valid for the propositions return them."
            "If not generate a new summary and a title based on the propositions."
            """
                Always return BOTH:
                - title
                - summary
            """
        ),
        (
            "user",
            "current_summary:{current_summary}\n current_title:{current_title}\n\n propositions:{propositions}",
        )
    ])
    summary_chain = summary_prompt_template | summary_llm

    chunk = get_chunk_by_id(chunk_id)

    current_summary = chunk["summary"]
    current_title = chunk["title"]
    current_propositions = chunk["propositions"]
    all_propositions = current_propositions + [proposition]
    chunk_meta = summary_chain.invoke(
        {
            "current_summary": current_summary,
            "current_title": current_title,
            "propositions": all_propositions
        }
    )

    chunk["summary"] = chunk_meta.summary
    chunk["title"] = chunk_meta.title
    chunk["propositions"] = all_propositions

    return chunk


def use_find_chunk_and_push_proposition(proposition):

    class ChunkID(BaseModel):
        chunk_id: int = Field(description="The chunk id.")

    allocation_llm = llm.with_structured_output(ChunkID)
    allocation_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """You assign propositions to semantic chunks.

                    Return the best matching chunk_id.
                    If none match, return a new chunk_id.
                    Return only the chunk_id.
                    """,),
                                (
                    "user",
                    """Proposition:
                    {proposition}

                    Existing chunks:
                    {chunks_summaries}
                """,
            ),
        ]
    )

    allocation_chain = allocation_prompt | allocation_llm

    chunks_summaries = {
        chunk["id"]: chunk["summary"] for chunk in chunks
    }

    best_chunk_id = allocation_chain.invoke(
        {"proposition": proposition, "chunks_summaries": chunks_summaries}
    ).chunk_id

    chunk = get_chunk_by_id(best_chunk_id)

    if chunk is None:
        create_new_chunk(best_chunk_id, proposition)
        return    

    add_proposition(best_chunk_id, proposition)

def get_chunk_by_id(chunk_id):
    for chunk in chunks:
        if chunk["id"] == chunk_id:
            return chunk
    return None

chunks

[]

In [75]:
documents
print("Documents type:", type(documents))
print("First element type:", type(documents[0]))
# for d in documents:
#     print(d.page_content)

Documents type: <class 'list'>
First element type: <class 'langchain_core.documents.base.Document'>


In [76]:

for d in documents:
    print("Documents type:", type(d))
    sentences = sent_tokenize(d.page_content)

    for s in sentences:
        use_find_chunk_and_push_proposition(s)
        print(s)
# print(json_data)
chunks

Documents type: <class 'langchain_core.documents.base.Document'>


NotImplementedError: 

In [13]:
parse_document("D:/Users/RouVin/Downloads/DEVELOPMENT PLAN - TOUR EAST FANTASY.pdf")

IMAGE PATH: D:/Users/RouVin/Downloads/DEVELOPMENT PLAN - TOUR EAST FANTASY.pdf
RESPONSE: DEVELOPMENT OF TOUR EAST FANTASY

Tour East Fantasy is a Tourism and Hospitality super app, providing tourists with ease of tourism information, promotions, and ecommerce capability to easily purchase tourism related products and services. Think of it as a Grab App for Tourism.

There will be 2 components to this platform

1. Tour East Fantasy 3js experience – This is the wow factor used for promoting the platform and to get partners on board. It's friendly and easy to navigate, to get visitors to explore and navigate. https://toureast.xctualyfe.com/

2. Tour East Fantasy Web App – This will encapsulate the entire experience and will lead the user to finding other products and services, as well as play hyper casual games to win prizes.

A table with three columns labeled Phase, Key Activity, and Duration. The table contains three rows of data:

Phase 1 has a duration of 4 weeks and includes the fol

["DEVELOPMENT OF TOUR EAST FANTASY\n\nTour East Fantasy is a Tourism and Hospitality super app, providing tourists with ease of tourism information, promotions, and ecommerce capability to easily purchase tourism related products and services. Think of it as a Grab App for Tourism.\n\nThere will be 2 components to this platform\n\n1. Tour East Fantasy 3js experience – This is the wow factor used for promoting the platform and to get partners on board. It's friendly and easy to navigate, to get visitors to explore and navigate. https://toureast.xctualyfe.com/\n\n2. Tour East Fantasy Web App – This will encapsulate the entire experience and will lead the user to finding other products and services, as well as play hyper casual games to win prizes.\n\nA table with three columns labeled Phase, Key Activity, and Duration. The table contains three rows of data:\n\nPhase 1 has a duration of 4 weeks and includes the following key activities:\n● Project Kick Off\n● Develop a web version of Tour